In [1]:
import os

!git clone https://github.com/uzbtrust/Uzbek-Operator-RAG-From-Scratch.git
os.chdir('Uzbek-Operator-RAG-From-Scratch')
!pip install -q tokenizers pyyaml torch faiss-cpu scikit-learn numpy

Cloning into 'Uzbek-Operator-RAG-From-Scratch'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 102 (delta 32), reused 94 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 90.90 KiB | 2.02 MiB/s, done.
Resolving deltas: 100% (32/32), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.0 MB/s eta 0:00:00:00:0100:01


In [2]:
import glob
import shutil

candidates = glob.glob('/kaggle/input/**/best_model.pt', recursive=True)
for c in candidates:
    print(f"topildi: {c} ({os.path.getsize(c)/1e6:.1f} MB)")

assert len(candidates) > 0, "best_model.pt topilmadi! Kaggle Input ga qo'shing"

os.makedirs('checkpoints/simcse', exist_ok=True)
shutil.copy(candidates[0], 'checkpoints/simcse/best_model.pt')
size = os.path.getsize('checkpoints/simcse/best_model.pt') / 1e6
print(f'checkpoint tayyor: {size:.1f} MB')

topildi: /kaggle/input/models/uzbtrust/uzbek-operator-rag-simcse/pytorch/best-model/1/best_model.pt (135.8 MB)
checkpoint tayyor: 135.8 MB


In [3]:
!python data/download_corpus.py --max-docs 50000
!python data/preprocess.py
!python tokenizer/train_tokenizer.py

2026-04-04 15:33:49,985 yangi shard: data/raw/shard_000.txt
2026-04-04 15:33:49,985 wikipedia yuklanmoqda...
2026-04-04 15:33:50,151 HTTP Request: HEAD https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-04 15:33:50,167 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/wikimedia/wikipedia/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/README.md "HTTP/1.1 200 OK"
2026-04-04 15:33:50,184 HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/wikimedia/wikipedia/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/README.md "HTTP/1.1 200 OK"
README.md: 131kB [00:00, 179MB/s]
2026-04-04 15:33:50,251 HTTP Request: HEAD https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/wikipedia.py "HTTP/1.1 404 Not Found"
2026-04-04 15:33:50,361 HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/wikimedia/wikipedia/wikimedia/wikipedia.py "HTTP/1

In [4]:
!python data/synthetic_qa_generator.py

import json
from collections import Counter

with open('data/synthetic_qa.json') as f:
    qa = json.load(f)

print(f'jami: {len(qa)} ta juftlik')
cats = Counter(item['category'] for item in qa)
for cat, count in sorted(cats.items()):
    print(f'  {cat}: {count}')

2026-04-04 15:35:22,291 1015 ta QA juftlik yaratildi -> data/synthetic_qa.json
2026-04-04 15:35:22,291   address: 91
2026-04-04 15:35:22,291   complaints: 103
2026-04-04 15:35:22,291   contact: 88
2026-04-04 15:35:22,291   documents: 105
2026-04-04 15:35:22,291   no_info: 15
2026-04-04 15:35:22,291   payment: 110
2026-04-04 15:35:22,291   pricing: 102
2026-04-04 15:35:22,291   promotions: 94
2026-04-04 15:35:22,291   services: 101
2026-04-04 15:35:22,291   tech_support: 109
2026-04-04 15:35:22,291   working_hours: 97
jami: 1015 ta juftlik
  address: 91
  complaints: 103
  contact: 88
  documents: 105
  no_info: 15
  payment: 110
  pricing: 102
  promotions: 94
  services: 101
  tech_support: 109
  working_hours: 97


In [5]:
import sys
sys.path.insert(0, os.getcwd())

from retriever.chunker import load_and_chunk
from retriever.hybrid_retriever import HybridRetriever

chunks = load_and_chunk('data/sample_operator.txt')
print(f'{len(chunks)} ta chunk')

retriever = HybridRetriever('configs/config.yaml', 'checkpoints/simcse/best_model.pt')
retriever.index(chunks)

test_questions = [
    "What are the working hours?",
    "How can I contact you?",
    "What payment methods do you accept?",
    "How much does the Premium Plan cost?",
    "What documents are required?",
]

for q in test_questions:
    results = retriever.search(q, top_k=3)
    print(f'\n--- {q} ---')
    for i, r in enumerate(results):
        src = r['source']
        sc = r['score']
        txt = r['chunk']['text'][:120].replace('\n', ' ')
        print(f'  [{i+1}] {src} score={sc:.4f} | {txt}')

2026-04-04 15:35:23,772 NumExpr defaulting to 4 threads.
2026-04-04 15:35:25,789 Loading faiss with AVX512 support.
2026-04-04 15:35:25,814 Successfully loaded faiss with AVX512 support.
2026-04-04 15:35:25,832 13 ta chunk yaratildi: data/sample_operator.txt


13 ta chunk


2026-04-04 15:35:26,709 model yuklandi: checkpoints/simcse/best_model.pt
2026-04-04 15:35:27,067 tfidf index yaratildi: 13 chunk, 474 feature
2026-04-04 15:35:27,607 faiss index yaratildi: 13 chunk, 512 dim
2026-04-04 15:35:27,608 hybrid index tayyor: 13 chunk



--- What are the working hours? ---
  [1] hybrid score=0.4350 | WORKING HOURS Office hours: Monday to Friday, 9:00 to 18:00 Saturday: 10:00 to 14:00 Sunday: Closed Call center: Availab
  [2] hybrid score=0.2006 | TECHNICAL SUPPORT Technical support phone: +998 71 200 00 02 Available: 24/7 Internet speed test: speedtest.uztelecom.uz
  [3] hybrid score=0.1526 | TARIFF PLANS Basic Plan: 50,000 UZS/month - 5 Mbps internet, 100 minutes calls Standard Plan: 100,000 UZS/month - 20 Mbp

--- How can I contact you? ---
  [1] hybrid score=0.2918 | CONTACT INFORMATION Phone: +998 71 123 45 67 Hotline: 1099 (free of charge) Email: info@uztelecom.uz Support email: supp
  [2] hybrid score=0.2297 | PAYMENT METHODS - Payme app - Click app - UzCard / Humo cards - PAYNET terminals - Cash at any office - Bank transfer Ba
  [3] hybrid score=0.1781 | COMPANY INFORMATION Company Name: UzTelecom Founded: 1992 Type: Telecommunications operator

--- What payment methods do you accept? ---
  [1] hybrid score=0.

In [6]:
import yaml
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
print("confidence_threshold:", cfg["retriever"]["confidence_threshold"])

from rag.confidence import ConfidenceChecker
confidence = ConfidenceChecker('configs/config.yaml')

test_questions = [
    ("How can I contact you?", True),
    ("What are the working hours?", True),
    ("What payment methods do you accept?", True),
    ("How much does the Premium Plan cost?", True),
    ("What documents are required?", True),
    ("Do you sell airplanes?", False),
    ("What is the capital of France?", False),
]

correct = 0
for q, should_pass in test_questions:
    results = retriever.search(q, top_k=3)
    passed, score = confidence.check(results)
    status = "OK" if passed else "FALLBACK"
    is_correct = passed == should_pass
    correct += int(is_correct)
    mark = "✓" if is_correct else "✗"
    print(f'[{status:8s}] score={score:.3f} {mark} | {q}')

print(f'\nconfidence accuracy: {correct}/{len(test_questions)}')

2026-04-04 15:35:27,712 confidence threshold: 0.2
2026-04-04 15:35:27,759 confidence past: 0.1795 < 0.2
2026-04-04 15:35:27,768 confidence past: 0.1713 < 0.2


confidence_threshold: 0.2
[OK      ] score=0.292 ✓ | How can I contact you?
[OK      ] score=0.435 ✓ | What are the working hours?
[OK      ] score=0.399 ✓ | What payment methods do you accept?
[OK      ] score=0.390 ✓ | How much does the Premium Plan cost?
[OK      ] score=0.290 ✓ | What documents are required?
[FALLBACK] score=0.179 ✓ | Do you sell airplanes?
[FALLBACK] score=0.171 ✓ | What is the capital of France?

confidence accuracy: 7/7


In [7]:
!python eval/evaluate.py \
    --knowledge data/sample_operator.txt \
    --qa-data data/synthetic_qa.json \
    --output eval/results.json

import json
with open('eval/results.json') as f:
    r = json.load(f)

print("=" * 50)
print("RETRIEVAL METRICS")
print("=" * 50)
for k, v in r["retrieval"].items():
    print(f"  {k}: {v}")

print()
print("=" * 50)
print("FALLBACK METRICS")
print("=" * 50)
for k, v in r["fallback"].items():
    print(f"  {k}: {v}")

print()
print(f"chunks: {r['config']['num_chunks']}, questions: {r['config']['num_questions']}")

2026-04-04 15:35:28,823 NumExpr defaulting to 4 threads.
2026-04-04 15:35:30,642 Loading faiss with AVX512 support.
2026-04-04 15:35:30,665 Successfully loaded faiss with AVX512 support.
2026-04-04 15:35:30,680 13 ta chunk yaratildi: data/sample_operator.txt
2026-04-04 15:35:30,680 bilim bazasi: 13 chunk
2026-04-04 15:35:30,682 test set: 1015 savol
2026-04-04 15:35:31,323 model yuklandi: checkpoints/simcse/best_model.pt
2026-04-04 15:35:31,633 tfidf index yaratildi: 13 chunk, 474 feature
2026-04-04 15:35:31,908 faiss index yaratildi: 13 chunk, 512 dim
2026-04-04 15:35:31,908 hybrid index tayyor: 13 chunk
2026-04-04 15:35:31,911 confidence threshold: 0.2
2026-04-04 15:35:39,322 === Retrieval Metrics ===
2026-04-04 15:35:39,322   mrr@5: 0.985
2026-04-04 15:35:39,322   ndcg@5: 0.985
2026-04-04 15:35:39,322   recall@5: 0.985
2026-04-04 15:35:39,322 === Fallback Metrics ===
2026-04-04 15:35:39,322   precision: 0.0
2026-04-04 15:35:39,322   recall: 0.0
2026-04-04 15:35:39,322   true_positive